# Vertex AI: Gemini 3.1 Flash Image Demo (Preview)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/gemini_3_1_flash_image_demo.ipynb)

This notebook demonstrates the **[Gemini 3.1 Flash Image](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models/gemini/3-1-flash-image)** model, which provides high-quality image generation with state-of-the-art prompt adherence and native multimodality. [Release Detail](https://docs.cloud.google.com/vertex-ai/docs/release-notes#February_26_2026)


In [ ]:
!pip install google-genai Pillow
from google.colab import auth
auth.authenticate_user()

from google import genai
from PIL import Image
from io import BytesIO
import os

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
client = genai.Client(vertexai=True, project=project_id, location="global")

### 2. Generate and Display Image

Gemini 3.1-flash-image handles generation and understanding in a single unified call.

In [ ]:
import time
import random

# Define beautiful, diverse prompts (Building, Flower, Animal)
prompts = [
    # Building (Solarpunk Aesthetic)
    "A stunning solarpunk vision of a futuristic city where skyscrapers are completely integrated vertical forests. Glass bubbles of flying electric vehicles navigate between lush green towers. Photorealistic 8k, warm golden hour lighting, biomechanical details, serene atmosphere.",

    # Flower (Abstract/Cinematic)
    "An intricate close-up of a fantastical biomechanical lotus flower. The petals are translucent, pulsing with cyan and magenta neon light, made of fine metallic filigree and organic filaments. Resting on a dark, reflective mirror-like water surface. Cinematic depth of field, octane render.",

    # Animal (Photorealistic 8K)
    "A majestic, highly detailed 8k portrait of an iridescent chameleon camouflaged on a vibrant mossy branch. Its skin has complex, prismatic scales. The eyes are hyper-realistic, capturing reflections. Soft bokeh background of a rainforest. Warm, natural lighting, macro photography."
]

# Fast Demo Config (1:1, 512px for maximum speed)
config = {
    "response_modalities": ["IMAGE"],
    "image_config": {
        "aspect_ratio": "1:1",
        "image_size": "512"
    }
}

print(f"Generating 3 diverse, beautiful images with gemini-3.1-flash-image-preview...")

for idx, prompt in enumerate(prompts):
    print(f"\n--- Generating Image {idx + 1}/{len(prompts)} ---")
    print(f"Prompt: '{prompt[:75]}...'")

    response = None
    max_retries = 3

    for retry_count in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-3.1-flash-image-preview",
                contents=[prompt],
                config=config
            )
            break
        except Exception as e:
            if "429" in str(e) or "500" in str(e):
                wait_time = (2 ** retry_count) + random.random()
                print(f"Quota hit. Retrying in {wait_time:.1f}s...")
                time.sleep(wait_time)
            else:
                print(f"Error: {e}")
                break

    if response:
        for part in response.candidates[0].content.parts:
            if part.inline_data:
                img = Image.open(BytesIO(part.inline_data.data))
                display(img)
                print(f"Success: Image {idx + 1} displayed.")
    else:
        print(f"Failed to generate image {idx + 1}.")